# Hello World
## Welcome to the Azure HDInsight Jupyter notebook

This is a first-step notebook on the cluster. After a short overview of how Spark is organised here, you will **inspect your Spark session**, then load **HVAC** sample data from `/HdiSamples/`, explore it as a **DataFrame**, save it as a **Hive table**, and run **SQL** through `spark.sql`.

## How Spark works on HDInsight (reminder)

**HDInsight** gives you a managed cluster. This Jupyter notebook runs on an edge or head node and talks to **Spark**. Spark splits work across **worker** machines: the process running your notebook (the **driver**) builds a **plan**, and **executors** run **tasks** in parallel on **partitions** of the data.

**Data is partitioned.** Large files are not read as one giant block in one place: Spark (and the underlying store such as HDFS or Azure Storage) splits the input into **chunks**. Each chunk can be processed on a different executor. That is how parallel speed-up appears.

An **RDD** (**resilient distributed dataset**) is simply a dataset **split into partitions** across the cluster so work can run **in parallel** on many machines.

**Lazy evaluation:** when you call `read.csv` or build a query, Spark often records **what** to do, not **everything at once**. Heavy work runs when you call an **action** (for example `count()`, `show()`, or writing a table).

The next section prints a few facts about **your** current session (version, master, app name).

## Inspect your Spark session

The HDInsight kernel already created `spark`. The **master** URL tells you whether you are on `yarn` (cluster), `local`, and so on. **App name** helps you find this job in the Spark UI.

In [ ]:
print("Spark version:", spark.version)
sc = spark.sparkContext
print("Master:", sc.master)
print("App name:", sc.appName)


### Imports

PySpark and Spark SQL are available on the HDInsight Jupyter kernel. Import the symbols you need for DataFrames and types.

In [ ]:
from pyspark.sql import *
from pyspark.sql.types import *


The **Spark session** (`spark`) is provided by the HDInsight kernel; you do not need to create it. The sample data is under `/HdiSamples/` on the cluster.

**Read the HVAC CSV** from the cluster path into a Spark DataFrame. The first row is used as the header and Spark infers column types.

In [ ]:
# Create a Spark DataFrame from sample data (this is not a pandas DataFrame)
csvFile = spark.read.csv(
    "/HdiSamples/HdiSamples/SensorSampleData/hvac/HVAC.csv",
    header=True,
    inferSchema=True,
)


**Row count:** check how many records are in the DataFrame.

In [ ]:
# How many rows in the DataFrame?
csvFile.count()


### HVAC dataset description

The **HVAC** (Heating, Ventilation, and Air Conditioning) sample data contains sensor readings from buildings: date and time, target and actual temperatures, system identifier, system age, and building ID. Below we explore the schema and a sample of the data before saving it as a Hive table.

In [ ]:
# Show the schema: column names and types inferred by Spark
csvFile.printSchema()


In [ ]:
# Preview the first 10 rows
csvFile.show(10)


In [ ]:
# Summary statistics for numeric columns
csvFile.describe().show()


### Save the DataFrame as a Hive table

We drop the table if it already exists, then write the DataFrame to the Hive metastore so we can query it with SQL. This persists the table for the cluster (unlike a temporary view).

All SQL in this notebook runs through **`spark.sql("...")`** (and usually `.show()` to print the result). That uses the same engine as the DataFrame API; there is no separate SQL-only cell type in this notebook.

In [ ]:
spark.sql("DROP TABLE IF EXISTS hvac_ss01shh")


In [ ]:
csvFile.write.saveAsTable("hvac_ss01shh")


**List tables** in the default database to confirm the HVAC table is present.

In [ ]:
spark.sql("SHOW TABLES").show()


**Describe the table** to see column names and types in the Hive catalog.

In [ ]:
spark.sql("DESCRIBE TABLE hvac_ss01shh").show()


**Example query:** temperature difference (target minus actual) for a given date. Positive values mean the system was heating; negative means cooling.

In [ ]:
query = """
SELECT `buildingID`, (targettemp - actualtemp) AS `temp_diff`, date
FROM hvac_ss01shh
WHERE date = '6/1/13'
"""
spark.sql(query).show()


**Total row count** in the Hive table (should match the DataFrame count).

In [ ]:
spark.sql("SELECT COUNT(*) AS Count FROM hvac_ss01shh").show()
